<a href="https://colab.research.google.com/github/KIMEUIJOON-KNU/Bi-directional/blob/main/naive_inverse_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # 두 번째 GPU만 보이게 설정 (0=첫 번째, 1=두 번째)

import torch
device = torch.device("cuda:0")  # 여기서 cuda:0은 "visible GPU 중 첫 번째" → 실제 GPU1
print("사용 중인 GPU:", torch.cuda.get_device_name(device))


print("총 GPU 개수:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

사용 중인 GPU: NVIDIA GeForce RTX 4080 SUPER
총 GPU 개수: 1
GPU 0: NVIDIA GeForce RTX 4080 SUPER


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import os
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
# 파장 정의
WAVELENGTHS = np.arange(380, 781, 1)
NUM_WAVELENGTHS = len(WAVELENGTHS) #401

# GitHub raw 주소 (파일들이 위치한 경로)
base_url = "https://raw.githubusercontent.com/KIMEUIJOON-KNU/Bi-directional/main/"

# 사용할 재료 목록
material_files = [
    ('SiO2', 'SiO2_380.csv'),
    ('WO3',  'WO3_380.csv'),
    ('Ag',   'Ag_380.csv')
]

material_data = {}
material_names = [info[0] for info in material_files]
print("재료 리스트:", material_names)

# 재료 인덱싱 매핑
material_names = sorted(set(material_names))
material_to_index = {name: i for i, name in enumerate(material_names)}
index_to_material = {i: name for name, i in material_to_index.items()}
print("재료 인덱스 매핑:", index_to_material)

# --- 재료 파일 불러오기 ---
for material_name, filename in material_files:
    file_url = base_url + filename
    print(f"처리 중: {filename} (materials: {material_name}) → {file_url}")
    try:
        data = pd.read_csv(file_url, header=None)
    except Exception as e:
        print(f"  🚫 오류 발생: {e}")
        continue

    if data.empty:
        print("  ⚠️ 경고: 파일이 비어있습니다.")
        continue

    wavelengths = data.iloc[:, 0].values  # 파장
    n = data.iloc[:, 1].values           # 실수부
    k = data.iloc[:, 2].values           # 허수부

    n_complex = n + 1j * k               # 복소 굴절률

    material_data[material_name] = (wavelengths, n_complex)
    print(f"로드 완료: {len(wavelengths)} points")

# --- 공기 굴절률 추가 ---
material_data['Air'] = np.ones(NUM_WAVELENGTHS, dtype=np.complex128)
print("Air 굴절률 추가 완료")


['SiO2', 'WO3', 'Ag']
{0: 'Ag', 1: 'SiO2', 2: 'WO3'}
처리 중: SiO2_380.csv (materials: SiO2)
  로드된 샘플 수: 401
처리 중: WO3_380.csv (materials: WO3)
  로드된 샘플 수: 401
처리 중: Ag_380.csv (materials: Ag)
  로드된 샘플 수: 401


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# (A) 환경 및 TMM 파라미터 준비
# ────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# (1) 파장 정의 (380~780 nm, 5 nm 간격 → 81개)
WAVELENGTHS = np.arange(380, 781, 1)            # [380, 385, ..., 780]
NUM_WAVELENGTHS = len(WAVELENGTHS)             # 81
λ_tensor_global = torch.tensor(WAVELENGTHS, dtype=torch.float64).to(device) * 1e-9  # [m 단위]

material_sequence = ['Ag', 'WO3', 'Ag']

#각 층(material_sequence)의 복소 n(λ) 배열을 numpy로부터 추출
n_list_np_arrays = [material_data[mat][1] for mat in material_sequence]  # 길이 3 리스트


#numpy → torch로 변환 & device로 이동 (dtype=torch.cdouble)
n_list_torch = torch.stack([
    torch.from_numpy(arr).to(torch.complex64) for arr in n_list_np_arrays
], dim=0).to(device)
print(n_list_torch.shape)

#    입사·출사 매질 굴절률 (예: 입사=SiO2, 출사=Air)
n_i_np = material_data['SiO2'][1]   # numpy complex
n_s_np = material_data['Air']       # numpy complex

n_i_torch = torch.from_numpy(n_i_np).to(torch.complex64).to(device)
n_s_torch = torch.from_numpy(n_s_np).to(torch.complex64).to(device)

# (4) 두께 그리드 (nm 단위)
d1_list_nm = np.arange(5, 41, 5)
d2_list_nm = np.arange(150,1001, 5)
d3_list_nm = np.arange(5, 41, 5)

print(len(d1_list_nm)*len(d2_list_nm)*len(d3_list_nm))
# (5) 샘플 저장용 리스트
all_d_tilde = []    # [(tilde1, tilde2, tilde3), …]
all_T_target = []   # [array([T(λ), …, T(λ)]), …]

Using device: cuda
torch.Size([3, 401])
10944


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# (A-1) TMMNetwork 정의 (클래스 이름, dtype 통일, calculate() 추가)
# ────────────────────────────────────────────────────────────────────────────
class TMMNetwork(nn.Module):
    def __init__(self, n_list_input, n_i_input, n_s_input, wavelengths_m_tensor):
        super().__init__()
        dtype_complex = torch.complex128
        dtype_float = torch.float64

        # 기준길이 L0 (이제 tilde는 안 쓰므로 없어도 무방하지만 유지 가능)
        n_list_real = n_list_input.real
        n_list_imag = -torch.abs(n_list_input.imag)
        self.register_buffer('n_list', torch.complex(n_list_real, n_list_imag))
        self.register_buffer('n_i', n_i_input.to(dtype_complex))
        self.register_buffer('n_s', n_s_input.to(dtype_complex))

        # 파수 k0
        k0 = 2 * torch.pi / torch.clamp(wavelengths_m_tensor, min=1e-20)
        self.register_buffer('k0', k0)  # 이제 k0_tilde 아님

        self.num_layers = self.n_list.shape[0]
        self.num_wavelengths = wavelengths_m_tensor.shape[0]

    def forward(self, thicknesses_nm):
        device = thicknesses_nm.device
        dtype_complex = torch.complex128
        imag_unit = torch.tensor(1j, dtype=dtype_complex, device=device)

        if thicknesses_nm.ndim == 1:
            thicknesses_nm = thicknesses_nm.unsqueeze(0)  # (1, 3)
        B = thicknesses_nm.shape[0]  # batch size

        # Convert to meters
        thicknesses_m = thicknesses_nm * 1e-9  # (B, 3)

        # (B, 3, 401): broadcasting layer × wavelength
        delta = thicknesses_m[:, :, None] * self.k0[None, None, :] * self.n_list[None, :, :]

        cosδ = torch.cos(delta)
        sinδ = torch.sin(delta)

        # 초기 M_total: (B, 401, 2, 2)
        M_total = torch.eye(2, dtype=dtype_complex, device=device).repeat(B, self.num_wavelengths, 1, 1)

        for j in range(self.num_layers):

            Yj = self.n_list[j] * 2.654e-3  # (401,)

            sin_j = sinδ[:, j, :]
            cos_j = cosδ[:, j, :]

            m11 = cos_j
            m12 = imag_unit * sin_j / Yj[None, :]
            m21 = imag_unit * Yj[None, :] * sin_j
            m22 = cos_j

            Mj = torch.stack([
                torch.stack([m11, m12], dim=-1),
                torch.stack([m21, m22], dim=-1)
            ], dim=-2)  # shape: (B, 401, 2, 2)

            M_total = torch.matmul(M_total, Mj)

        Y_in = self.n_i * 2.654e-3
        Y_out = self.n_s * 2.654e-3

        B_ = M_total[:, :, 0, 0] + M_total[:, :, 0, 1] * Y_out
        C_ = M_total[:, :, 1, 0] + M_total[:, :, 1, 1] * Y_out
        denom = Y_in * B_ + C_
        t1 = (2 * Y_in) / denom  # (B, 401)

        T = (torch.real(self.n_s) / torch.real(self.n_i)) * torch.abs(t1) ** 2
        return T.to(torch.float64)  # (B, 401)
# ────────────────────────────────────────────────────────────────────────────
# (A-2) 단일 시험용 스펙트럼 계산 (디버깅 / 시각화)
# ────────────────────────────────────────────────────────────────────────────
model = TMMNetwork(n_list_torch, n_i_torch, n_s_torch, λ_tensor_global).to(device)

In [ ]:
import os
import random
import numpy as np
import torch

# ------------------------------
# 시드 고정 (재현성 확보)
# ------------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)  # 시드 고정 실행

In [ ]:


import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import itertools

print("Generating (d -> T_target) pairs using efficient batch processing...")

# 1. 모든 두께 조합을 미리 생성합니다.
thickness_combinations = list(itertools.product(d1_list_nm, d2_list_nm, d3_list_nm))
all_d_nm_batch = torch.tensor(thickness_combinations, dtype=torch.float64) # shape: (N, 3)


with torch.no_grad():
    all_T_target_batch = model(all_d_nm_batch.to(device))

# 3. 후속 처리를 위해 결과를 NumPy 배열로 변환합니다.
all_d = all_d_nm_batch.numpy()
all_T_target = all_T_target_batch.cpu().numpy()

# ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ두께 표준화ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ
THICKNESS_MIN = torch.tensor(np.min(all_d, axis=0), dtype=torch.float64, device=device)
THICKNESS_MAX = torch.tensor(np.max(all_d, axis=0), dtype=torch.float64, device=device)

def standardize_thickness(d_nm_array):
    d_nm_array = torch.tensor(d_nm_array, dtype=torch.float64, device=THICKNESS_MIN.device) if not isinstance(d_nm_array, torch.Tensor) else d_nm_array.to(dtype=torch.float64, device=THICKNESS_MIN.device)
    return (d_nm_array - THICKNESS_MIN) / (THICKNESS_MAX - THICKNESS_MIN)

def destandardize_thickness(d_norm_array):
    d_norm_array = d_norm_array.clone().to(dtype=torch.float64, device=THICKNESS_MIN.device)
    return d_norm_array * (THICKNESS_MAX - THICKNESS_MIN) + THICKNESS_MIN

all_d_norm = standardize_thickness(all_d)
# d_nm_check = destandardize_thickness(all_d_norm) # 필요 시 확인용

# ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ스펙트럼 표준화ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ
print("Total samples:", all_d.shape[0])

SPECTRUM_MIN = torch.tensor(np.min(all_T_target, axis=0), dtype=torch.float64).to(device)
SPECTRUM_MAX = torch.tensor(np.max(all_T_target, axis=0), dtype=torch.float64).to(device)

def standardize_spectrum(T_array, eps=1e-8):
    T_tensor = torch.tensor(T_array, dtype=torch.float64).to(device)
    return (T_tensor - SPECTRUM_MIN) / (SPECTRUM_MAX - SPECTRUM_MIN + eps)

def destandardize_spectrum(T_std_array):
    return T_std_array * (SPECTRUM_MAX - SPECTRUM_MIN) + SPECTRUM_MIN

all_T_target_std = standardize_spectrum(all_T_target)

# --------------------------------------------------------------------------
# (C) Dataset / DataLoader 구축 (기존 코드와 동일)
# --------------------------------------------------------------------------
class BiTMMNormalizedDataset(Dataset):
    def __init__(self, d_norm_array, T_array):
        self.d_norm = d_norm_array.clone().to(dtype=torch.float64)
        self.T_spec = T_array.clone().detach().to(dtype=torch.float64) if isinstance(T_array, torch.Tensor) else torch.tensor(T_array, dtype=torch.float64)

    def __len__(self):
        return self.d_norm.shape[0]

    def __getitem__(self, idx):
        return {
            'd_norm': self.d_norm[idx],
            'T_target': self.T_spec[idx]
        }

batch_size = 32

# 시드 고정
seed = 42
g = torch.Generator().manual_seed(seed)

# Dataset 생성
dataset = BiTMMNormalizedDataset(all_d_norm, all_T_target_std)

# 전체 길이 및 split 비율
total_size = len(dataset)
train_size = int(0.9 * total_size)
val_size   = int(0.05 * total_size)
test_size  = total_size - train_size - val_size

print(f"Total: {total_size}, Train: {train_size}, Val: {val_size}, Test: {test_size}")

# Dataset 분할
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=g
)

# DataLoader 정의
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=0, generator=g)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

Generating (d -> T_target) pairs using efficient batch processing...
Total samples: 10944
Total: 10944, Train: 9849, Val: 547, Test: 548


In [ ]:
# @title
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
import optuna
from torch.utils.data import random_split, DataLoader
from tqdm import tqdm


def objective(trial):
    """Optuna가 한 세트의 하이퍼파라미터를 시험(trial)하는 함수"""
    def seed_everything(seed: int = 42):
        import random, numpy as np
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    SEED = 42
    seed_everything(SEED)

    # --- 1. 탐색할 하이퍼파라미터 정의 ---
    lr = trial.suggest_float("lr", 5e-5, 1e-4)
    alpha = trial.suggest_float("alpha", 0.05, 0.3, log=True)
    n_layers = 4
    #n_layers = trial.suggest_int("n_layers", 4, 5)
    n_units = trial.suggest_categorical("n_units", [768, 1024])
    batch_size = trial.suggest_categorical("batch_size", [32,64,128])
    #beta = trial.suggest_int("beta", 10, 30)
    num_epochs = trial.suggest_int("num_epochs", 150, 300)

    # 보기 좋게 출력합니다.
    print("\n" + "="*60)
    print(f"▶ Trial {trial.number} 시작 (적용된 하이퍼파라미터):")
    # trial.params를 직접 사용하여 모든 제안된 파라미터를 출력합니다.
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    print("="*60)
    # ▲▲▲▲▲ 수정된 부분 ▲▲▲▲▲

    # --- 2. 데이터 분할 및 DataLoader 생성 ---
    g = torch.Generator().manual_seed(SEED)
    train_dataset, val_dataset, _ = random_split(
        dataset, [train_size, val_size, test_size], generator=g
    )
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, generator=g
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=0
    )

    # --- 3. InverseNet 정의 ---
    class InverseNet(nn.Module):
        def __init__(self, input_dim, output_dim, num_hidden_layers, units):
            super().__init__()
            layers = [nn.Linear(input_dim, units), nn.LeakyReLU()]
            for _ in range(num_hidden_layers):
                layers += [nn.Linear(units, units), nn.LeakyReLU()]
            layers += [nn.Linear(units, output_dim), nn.Sigmoid()]
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x)

    num_layers_output = len(material_sequence)
    inverse_net = InverseNet(
        NUM_WAVELENGTHS, num_layers_output, n_layers, n_units
    ).to(device).to(torch.float64)
    optimizer = optim.Adam(inverse_net.parameters(), lr=lr)
    epoch_pbar = tqdm(range(1, num_epochs + 1), desc="Training Progress")

    # --- 4. 학습 및 검증 루프 ---
    for epoch in epoch_pbar:
        inverse_net.train()
        epoch_loss = 0.0
        for batch_idx, batch in enumerate(train_loader):
            T_target = batch['T_target'].to(device)
            d_norm_true = batch['d_norm'].to(device)
            optimizer.zero_grad()

            x = T_target.squeeze(1)

            d_norm_pred = inverse_net(x)
            loss = F.mse_loss(d_norm_pred, d_norm_true.squeeze(1))
            loss.backward()
            #torch.nn.utils.clip_grad_norm_(inverse_net.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        inverse_net.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for val_batch in val_loader:
                T_val = val_batch['T_target'].to(device)
                d_norm_val = val_batch['d_norm'].to(device)
                x_val = T_val.squeeze(1).to(torch.float64)
                d_pred_val = inverse_net(x_val)
                val_loss = F.mse_loss(d_pred_val, d_norm_val.squeeze(1))
                val_loss_total += val_loss.item()

        avg_train_loss = epoch_loss / len(train_loader)
        avg_val_loss = val_loss_total / len(val_loader)
        epoch_pbar.set_postfix(train_loss=f"{avg_train_loss:.6f}", val_loss=f"{avg_val_loss:.6f}")

        trial.report(avg_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return avg_val_loss

if __name__ == "__main__":
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
        sampler=optuna.samplers.TPESampler(seed=43)
    )
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study.optimize(objective, n_trials=300, n_jobs=1)

    print("\n\n========================================================")
    print("                      탐색 완료!                      ")
    print("========================================================")
    print("최적의 Trial 번호:", study.best_trial.number)
    print("최적의 Loss (Value):", study.best_trial.value)
    print("\n최적의 하이퍼파라미터 (Best Params):")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

def seed_everything(seed: int = 42):
    import random, numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 시드 고정
SEED = 42
seed_everything(SEED)
alpha = 0.1489072672242894
lr = 7.086817257648059e-05
n_layers = 4
n_units = 1538
batch_size = 64
num_epochs = 1000



 # 원하는 epoch 수로 변경 가능

# --- 2. 데이터 분할 및 DataLoader 생성 ---
g = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size], generator=g
)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, generator=g
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=0
)

# --- 3. InverseNet 정의 ---
class InverseNet(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, units):
        super().__init__()
        layers = [nn.Linear(input_dim, units), nn.LeakyReLU()]
        for _ in range(num_hidden_layers):
            layers += [nn.Linear(units, units), nn.LeakyReLU()]
        layers += [nn.Linear(units, output_dim), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

num_layers_output = len(material_sequence)
inverse_net = InverseNet(
    NUM_WAVELENGTHS, num_layers_output, n_layers, n_units
).to(device).to(torch.float64)

optimizer = optim.Adam(inverse_net.parameters(), lr=lr)

# --- 4. 학습 루프 ---
for epoch in range(1, num_epochs + 1):
    # Train
    inverse_net.train()
    epoch_loss = 0.0
    for batch in train_loader:
        T_target = batch['T_target'].to(device)
        d_norm_true = batch['d_norm'].to(device)

        optimizer.zero_grad()
        x = T_target.squeeze(1)
        d_norm_pred = inverse_net(x)
        loss = F.mse_loss(d_norm_pred, d_norm_true.squeeze(1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    # Validation
    inverse_net.eval()
    val_loss_total = 0.0
    with torch.no_grad():
        for val_batch in val_loader:
            T_val = val_batch['T_target'].to(device)
            d_norm_val = val_batch['d_norm'].to(device)
            x_val = T_val.squeeze(1).to(torch.float64)
            d_pred_val = inverse_net(x_val)
            val_loss = F.mse_loss(d_pred_val, d_norm_val.squeeze(1))
            val_loss_total += val_loss.item()

    avg_val_loss = val_loss_total / len(val_loader)

    # 각 epoch 결과 출력
    print(f"[Epoch {epoch:03d}] Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")

# --- 5. 테스트 세트 평가 ---
inverse_net.eval()
test_loss_total = 0.0
with torch.no_grad():
    for test_batch in test_loader:
        T_test = test_batch['T_target'].to(device)
        d_norm_test = test_batch['d_norm'].to(device)
        x_test = T_test.squeeze(1).to(torch.float64)
        d_pred_test = inverse_net(x_test)
        test_loss = F.mse_loss(d_pred_test, d_norm_test.squeeze(1))
        test_loss_total += test_loss.item()

avg_test_loss = test_loss_total / len(test_loader)
print(f"최종 Test Loss: {avg_test_loss:.6f}")


[Epoch 001] Train Loss: 0.060942, Val Loss: 0.038645
[Epoch 002] Train Loss: 0.035349, Val Loss: 0.032847
[Epoch 003] Train Loss: 0.029436, Val Loss: 0.025090
[Epoch 004] Train Loss: 0.023881, Val Loss: 0.020340
[Epoch 005] Train Loss: 0.021691, Val Loss: 0.019627
[Epoch 006] Train Loss: 0.020616, Val Loss: 0.021456
[Epoch 007] Train Loss: 0.019735, Val Loss: 0.017540
[Epoch 008] Train Loss: 0.017446, Val Loss: 0.016923
[Epoch 009] Train Loss: 0.016363, Val Loss: 0.015890
[Epoch 010] Train Loss: 0.014073, Val Loss: 0.010734
[Epoch 011] Train Loss: 0.009580, Val Loss: 0.008323
[Epoch 012] Train Loss: 0.008804, Val Loss: 0.010998
[Epoch 013] Train Loss: 0.007251, Val Loss: 0.006632
[Epoch 014] Train Loss: 0.006026, Val Loss: 0.007396
[Epoch 015] Train Loss: 0.005926, Val Loss: 0.005655
[Epoch 016] Train Loss: 0.005014, Val Loss: 0.004035
[Epoch 017] Train Loss: 0.006171, Val Loss: 0.004895
[Epoch 018] Train Loss: 0.004596, Val Loss: 0.003632
[Epoch 019] Train Loss: 0.002737, Val Loss: 0.

In [ ]:
d_test_nm = torch.tensor([[20.0, 750.0, 20.0]], dtype=torch.float64).to(device)

with torch.no_grad():
    T_test_pred = model(d_test_nm)   # shape (1, 401)
inverse_net.eval()
with torch.no_grad():
    d_pred_norm = inverse_net(T_test_pred.to(torch.float64))  # 모델 출력 (정규화된 두께)
    d_pred_nm = destandardize_thickness(d_pred_norm)
print("입력한 실제 두께 (nm):", d_test_nm.cpu().numpy())
print("모델이 예측한 두께 (nm):", d_pred_nm.cpu().numpy())

입력한 실제 두께 (nm): [[ 20. 750.  20.]]
모델이 예측한 두께 (nm): [[ 18.50207406 743.40154403  21.78137913]]


In [ ]:
# 1. 테스트할 두께 직접 정의 (nm 단위)
d_test_nm = torch.tensor([[20.0, 562.0, 20.0]], dtype=torch.float64).to(device)  # shape (1,3)

# 2. TMM 모델로 스펙트럼 계산
with torch.no_grad():
    T_test_pred = model(d_test_nm)   # shape (1, 401)

# 3. InverseNet으로 두께 예측
inverse_net.eval()
with torch.no_grad():
    d_pred_norm = inverse_net(T_test_pred.to(torch.float64))  # 모델 출력 (정규화된 두께)
    d_pred_nm = destandardize_thickness(d_pred_norm)          # 역정규화해서 nm 단위 복원

print("입력한 실제 두께 (nm):", d_test_nm.cpu().numpy())
print("모델이 예측한 두께 (nm):", d_pred_nm.cpu().numpy())

# 4. 예측 두께로 다시 TMM 돌려서 재구성 스펙트럼 확인
with torch.no_grad():
    T_reconstructed = model(d_pred_nm.to(device))

# 5. 시각화
plt.figure(figsize=(10,5))
x_wavelengths = λ_tensor_global.cpu().numpy() * 1e9

plt.plot(x_wavelengths, T_test_pred.squeeze().cpu().numpy(),
         label="Original TMM Spectrum", linewidth=2)
plt.plot(x_wavelengths, T_reconstructed.squeeze().cpu().numpy(),
         'r--', label="Reconstructed Spectrum from Inverse Prediction", linewidth=2)

plt.xlabel("Wavelength (nm)")
plt.ylabel("Transmittance")
plt.title("Original vs Reconstructed Spectrum")
plt.legend()
plt.grid(True)
plt.ylim(0,1)
plt.show()


NameError: name 'torch' is not defined